In [ ]:
# Setup Google Colab if running in Colab environment
import sys
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    project_path = '/content/drive/MyDrive/xai-dark-matter-localization'
    sys.path.insert(0, project_path)
except ImportError:
    project_path = '/'.join(os.getcwd().split('/')[:-1])
    sys.path.insert(0, project_path)

os.chdir(project_path)
print(f"✓ Working directory: {os.getcwd()}")

In [ ]:
import numpy as np
import pandas as pd
import h5py
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, optimizers
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

print("TensorFlow version:", tf.__version__)
print("GPU available:", len(tf.config.list_physical_devices('GPU')) > 0)

In [ ]:
# Configuration
RESOLUTION = 512
DATA_DIR = Path('data/processed/TNG-DM-XAI-v1')
PREPROCESSED_DIR = DATA_DIR / 'preprocessed'
MODELS_DIR = DATA_DIR / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Load images from HDF5
print("Loading preprocessed images...")
with h5py.File(PREPROCESSED_DIR / f'images_{RESOLUTION}_preprocessed.h5', 'r') as f:
    X_train = f['X_train'][:]
    X_val = f['X_val'][:]
    X_test = f['X_test'][:]

# Load masks from HDF5
print("Loading masks...")
with h5py.File(PREPROCESSED_DIR / f'masks_{RESOLUTION}.h5', 'r') as f:
    M_train = f['masks_train'][:]
    M_val = f['masks_val'][:]
    M_test = f['masks_test'][:]

print(f"\n✓ Data loaded successfully:")
print(f"  X_train: {X_train.shape}, M_train: {M_train.shape}")
print(f"  X_val: {X_val.shape}, M_val: {M_val.shape}")
print(f"  X_test: {X_test.shape}, M_test: {M_test.shape}")

In [ ]:
def build_unet(input_shape=(512, 512, 1)):
    """Build U-Net architecture for dark matter localization"""
    
    inputs = layers.Input(shape=input_shape)
    
    # Encoder
    c1 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D((2, 2))(c1)
    
    c2 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D((2, 2))(c2)
    
    c3 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(p2)
    c3 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c3)
    p3 = layers.MaxPooling2D((2, 2))(c3)
    
    # Bottleneck
    c4 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(p3)
    c4 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(c4)
    
    # Decoder
    u3 = layers.UpSampling2D((2, 2))(c4)
    u3 = layers.concatenate([u3, c3])
    c5 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(u3)
    c5 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c5)
    
    u2 = layers.UpSampling2D((2, 2))(c5)
    u2 = layers.concatenate([u2, c2])
    c6 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(u2)
    c6 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c6)
    
    u1 = layers.UpSampling2D((2, 2))(c6)
    u1 = layers.concatenate([u1, c1])
    c7 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(u1)
    c7 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(c7)
    
    # Output layer
    outputs = layers.Conv2D(1, (1, 1), activation='sigmoid')(c7)
    
    model = models.Model(inputs=[inputs], outputs=[outputs])
    return model

# Build model
print("Building U-Net model...")
model = build_unet(input_shape=(RESOLUTION, RESOLUTION, 1))
print(f"✓ Model created with {model.count_params():,} parameters")
print(model.summary())

In [ ]:
# Custom loss functions
def dice_loss(y_true, y_pred):
    """Dice loss for binary segmentation"""
    smooth = 1.
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return 1. - (2. * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

def iou_loss(y_true, y_pred):
    """Jaccard/IoU loss for binary segmentation"""
    smooth = 1.
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - intersection
    return 1. - (intersection + smooth) / (union + smooth)

# Custom metrics
def dice_coefficient(y_true, y_pred):
    """Dice similarity coefficient"""
    smooth = 1.
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

def iou_metric(y_true, y_pred):
    """Intersection over Union metric"""
    smooth = 1.
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

# Compile model
print("Compiling model...")
optimizer = optimizers.Adam(learning_rate=1e-3)
model.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',  # Can also use dice_loss or iou_loss
    metrics=['accuracy', dice_coefficient, iou_metric]
)
print("✓ Model compiled successfully")

In [ ]:
# Training callbacks
checkpoint = callbacks.ModelCheckpoint(
    str(MODELS_DIR / 'model_best.h5'),
    monitor='val_iou_metric',
    mode='max',
    save_best_only=True,
    verbose=1
)

early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

class ProgressCallback(callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:3d} - "
                  f"Loss: {logs['loss']:.4f}, "
                  f"Val Loss: {logs['val_loss']:.4f}, "
                  f"Dice: {logs['dice_coefficient']:.4f}, "
                  f"Val Dice: {logs['val_dice_coefficient']:.4f}, "
                  f"IoU: {logs['iou_metric']:.4f}, "
                  f"Val IoU: {logs['val_iou_metric']:.4f}")

progress = ProgressCallback()

# Training configuration
BATCH_SIZE = 8
EPOCHS = 100

print(f"\n✓ Training configuration:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Train samples: {len(X_train)}")
print(f"  Val samples: {len(X_val)}")
print(f"  Test samples: {len(X_test)}")

In [ ]:
# Train the model
print("\n" + "="*80)
print("Starting Model Training")
print("="*80 + "\n")

history = model.fit(
    X_train, M_train,
    validation_data=(X_val, M_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[checkpoint, early_stopping, reduce_lr, progress],
    verbose=0
)

print("\n✓ Training completed!")
print(f"  Final train loss: {history.history['loss'][-1]:.4f}")
print(f"  Final val loss: {history.history['val_loss'][-1]:.4f}")
print(f"  Best val IoU: {max(history.history['val_iou_metric']):.4f}")

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss
axes[0, 0].plot(history.history['loss'], label='Train', linewidth=2)
axes[0, 0].plot(history.history['val_loss'], label='Val', linewidth=2)
axes[0, 0].set_xlabel('Epoch', fontsize=11)
axes[0, 0].set_ylabel('Loss (Binary Crossentropy)', fontsize=11)
axes[0, 0].set_title('Training & Validation Loss', fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Accuracy
axes[0, 1].plot(history.history['accuracy'], label='Train', linewidth=2)
axes[0, 1].plot(history.history['val_accuracy'], label='Val', linewidth=2)
axes[0, 1].set_xlabel('Epoch', fontsize=11)
axes[0, 1].set_ylabel('Accuracy', fontsize=11)
axes[0, 1].set_title('Training & Validation Accuracy', fontsize=12, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Dice Coefficient
axes[1, 0].plot(history.history['dice_coefficient'], label='Train', linewidth=2)
axes[1, 0].plot(history.history['val_dice_coefficient'], label='Val', linewidth=2)
axes[1, 0].set_xlabel('Epoch', fontsize=11)
axes[1, 0].set_ylabel('Dice Coefficient', fontsize=11)
axes[1, 0].set_title('Training & Validation Dice', fontsize=12, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# IoU Metric
axes[1, 1].plot(history.history['iou_metric'], label='Train', linewidth=2)
axes[1, 1].plot(history.history['val_iou_metric'], label='Val', linewidth=2)
axes[1, 1].set_xlabel('Epoch', fontsize=11)
axes[1, 1].set_ylabel('IoU (Jaccard)', fontsize=11)
axes[1, 1].set_title('Training & Validation IoU', fontsize=12, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(MODELS_DIR / 'training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Training history plot saved to {MODELS_DIR / 'training_history.png'}")

In [ ]:
# Load best model and evaluate on test set
print("\n" + "="*80)
print("Evaluating on Test Set")
print("="*80 + "\n")

# Load best model
model_best = keras.models.load_model(
    str(MODELS_DIR / 'model_best.h5'),
    custom_objects={'dice_coefficient': dice_coefficient, 'iou_metric': iou_metric}
)

# Evaluate on test set
test_loss, test_acc, test_dice, test_iou = model_best.evaluate(X_test, M_test, verbose=0)

print(f"Test Set Metrics:")
print(f"  Loss: {test_loss:.4f}")
print(f"  Accuracy: {test_acc:.4f}")
print(f"  Dice Coefficient: {test_dice:.4f}")
print(f"  IoU (Jaccard): {test_iou:.4f}")

# Generate predictions
print("\nGenerating predictions on test set...")
predictions = model_best.predict(X_test, verbose=0)

# Convert predictions to binary masks (threshold at 0.5)
predictions_binary = (predictions > 0.5).astype(np.float32)

# Calculate pixel-level metrics
from sklearn.metrics import confusion_matrix
y_true_flat = M_test.flatten()
y_pred_flat = predictions_binary.flatten()

tn, fp, fn, tp = confusion_matrix(y_true_flat, y_pred_flat).ravel()
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = sensitivity

print(f"\nPixel-level Metrics:")
print(f"  Sensitivity (Recall): {sensitivity:.4f}")
print(f"  Specificity: {specificity:.4f}")
print(f"  Precision: {precision:.4f}")

# Save metrics to CSV
metrics_df = pd.DataFrame({
    'Metric': ['Loss', 'Accuracy', 'Dice', 'IoU', 'Sensitivity', 'Specificity', 'Precision'],
    'Test_Value': [test_loss, test_acc, test_dice, test_iou, sensitivity, specificity, precision]
})
metrics_df.to_csv(MODELS_DIR / 'test_metrics.csv', index=False)
print(f"\n✓ Test metrics saved to {MODELS_DIR / 'test_metrics.csv'}")

In [ ]:
# Visualize predictions on test set
n_samples = 9
fig, axes = plt.subplots(n_samples, 3, figsize=(12, 14))

indices = np.random.choice(len(X_test), n_samples, replace=False)

for i, idx in enumerate(indices):
    # Original image
    axes[i, 0].imshow(X_test[idx, :, :, 0], cmap='gray')
    axes[i, 0].set_title('Input Image' if i == 0 else '', fontweight='bold')
    axes[i, 0].axis('off')
    
    # Ground truth mask
    axes[i, 1].imshow(M_test[idx, :, :, 0], cmap='gray')
    axes[i, 1].set_title('Ground Truth' if i == 0 else '', fontweight='bold')
    axes[i, 1].axis('off')
    
    # Prediction
    axes[i, 2].imshow(predictions[idx, :, :, 0], cmap='gray')
    axes[i, 2].set_title('Prediction' if i == 0 else '', fontweight='bold')
    axes[i, 2].axis('off')

plt.tight_layout()
plt.savefig(MODELS_DIR / 'sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Sample predictions saved to {MODELS_DIR / 'sample_predictions.png'}")

# Calculate per-image Dice scores
dice_scores = []
iou_scores = []

for i in range(len(X_test)):
    gt = M_test[i].flatten()
    pred = predictions[i].flatten()
    pred_binary = (pred > 0.5).astype(np.float32)
    
    # Dice
    intersection = np.sum(gt * pred_binary)
    dice = (2.0 * intersection + 1e-6) / (np.sum(gt) + np.sum(pred_binary) + 1e-6)
    dice_scores.append(dice)
    
    # IoU
    union = np.sum(gt) + np.sum(pred_binary) - intersection
    iou = (intersection + 1e-6) / (union + 1e-6)
    iou_scores.append(iou)

print(f"\nPer-Image Metrics:")
print(f"  Mean Dice: {np.mean(dice_scores):.4f} ± {np.std(dice_scores):.4f}")
print(f"  Mean IoU: {np.mean(iou_scores):.4f} ± {np.std(iou_scores):.4f}")

In [ ]:
# Save final model and training artifacts
print("\n" + "="*80)
print("Saving Model & Training Artifacts")
print("="*80 + "\n")

# Save model in multiple formats
# 1. HDF5 format (already saved as best model during training)
print("✓ Best model saved as: model_best.h5")

# 2. SavedModel format (for TensorFlow Serving)
try:
    model_best.save(str(MODELS_DIR / 'model_savedmodel'))
    print("✓ SavedModel format saved")
except Exception as e:
    print(f"⚠ SavedModel format failed: {e}")

# 3. TensorFlow Lite format (for mobile/edge)
try:
    converter = tf.lite.TFLiteConverter.from_keras_model(model_best)
    tflite_model = converter.convert()
    with open(MODELS_DIR / 'model.tflite', 'wb') as f:
        f.write(tflite_model)
    print("✓ TensorFlow Lite model saved")
except Exception as e:
    print(f"⚠ TensorFlow Lite conversion failed: {e}")

# Save training history as JSON
import json
history_dict = {
    'loss': [float(x) for x in history.history['loss']],
    'val_loss': [float(x) for x in history.history['val_loss']],
    'accuracy': [float(x) for x in history.history['accuracy']],
    'val_accuracy': [float(x) for x in history.history['val_accuracy']],
    'dice_coefficient': [float(x) for x in history.history['dice_coefficient']],
    'val_dice_coefficient': [float(x) for x in history.history['val_dice_coefficient']],
    'iou_metric': [float(x) for x in history.history['iou_metric']],
    'val_iou_metric': [float(x) for x in history.history['val_iou_metric']]
}
with open(MODELS_DIR / 'training_history.json', 'w') as f:
    json.dump(history_dict, f, indent=2)
print("✓ Training history saved as JSON")

# Save per-image metrics
results_df = pd.DataFrame({
    'Image_ID': np.arange(len(X_test)),
    'Dice': dice_scores,
    'IoU': iou_scores
})
results_df.to_csv(MODELS_DIR / 'per_image_metrics.csv', index=False)
print("✓ Per-image metrics saved")

print(f"\n🚀 All models and artifacts saved to: {MODELS_DIR}")
print(f"\n📊 Summary of saved files:")
print(f"  - model_best.h5: Best model checkpoint")
print(f"  - model_savedmodel/: TensorFlow SavedModel format")
print(f"  - model.tflite: TensorFlow Lite model")
print(f"  - training_history.png: Training curves plot")
print(f"  - training_history.json: Training metrics history")
print(f"  - sample_predictions.png: Visual predictions")
print(f"  - test_metrics.csv: Test set metrics")
print(f"  - per_image_metrics.csv: Per-image Dice and IoU scores")

# Model Training
Train CNN for dark matter localization using preprocessed TNG-100 dataset

## 1. Setup & Configuration

In [ ]:
# Add code here

## 2. Load Preprocessed Data

In [ ]:
# Add code here

## 3. Define Model Architecture

In [ ]:
# Add code here

## 4. Training Configuration

In [ ]:
# Add code here

## 5. Train Model

In [ ]:
# Add code here

## 6. Evaluate Model

In [ ]:
# Add code here

## 7. Save Model

In [ ]:
# Add code here